In [ ]:
%pip install polars pyarrow

In [ ]:
from pathlib import Path
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "crsp"
    / "01_crsp_sp500_daily_2003_2025.csv"
)

OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "02_crsp_sp500_monthly_2003_2025.parquet"
)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

print(f"Polars version: {pl.__version__}")
print(f"Raw file exists: {RAW_FILE.exists()}")
print(f"Raw file: {RAW_FILE}")
print(f"Output file: {OUTPUT_FILE}")

if not RAW_FILE.exists():
    raise FileNotFoundError(f"Raw file not found: {RAW_FILE}")

In [ ]:
schema_overrides = {
    "INDNO": pl.Int64,
    "MbrStartDt": pl.Utf8,
    "MbrEndDt": pl.Utf8,
    "PERMNO": pl.Int64,
    "HdrCUSIP": pl.Utf8,
    "PERMCO": pl.Int64,
    "SICCD": pl.Int64,
    "CUSIP": pl.Utf8,
    "Ticker": pl.Utf8,
    "ShareType": pl.Utf8,
    "SecurityType": pl.Utf8,
    "SecuritySubType": pl.Utf8,
    "USIncFlg": pl.Utf8,
    "IssuerType": pl.Utf8,
    "PrimaryExch": pl.Utf8,
    "ConditionalType": pl.Utf8,
    "TradingStatusFlg": pl.Utf8,
    "DlyCalDt": pl.Utf8,
    "DlyPrc": pl.Float64,
    "DlyPrcFlg": pl.Utf8,
    "DlyCap": pl.Float64,
    "DlyCapFlg": pl.Utf8,
    "DlyRet": pl.Float64,
    "DlyRetx": pl.Float64,
    "DlyRetMissFlg": pl.Utf8,
    "DlyRetDurFlg": pl.Utf8,
    "DlyOrdDivAmt": pl.Float64,
    "DlyNonOrdDivAmt": pl.Float64,
    "DlyFacPrc": pl.Float64,
    "DlyDistRetFlg": pl.Utf8,
    "DlyVol": pl.Float64,
    "DlyClose": pl.Float64,
    "DlyLow": pl.Float64,
    "DlyHigh": pl.Float64,
    "DlyBid": pl.Float64,
    "DlyAsk": pl.Float64,
    "DlyOpen": pl.Float64,
    "DlyNumTrd": pl.Int64,
    "DlyMMCnt": pl.Int64,
    "DlyCumFacPr": pl.Float64,
    "DlyCumFacShr": pl.Float64,
    "ShrOut": pl.Float64,
}

print(f"Number of schema definitions: {len(schema_overrides)}")

In [ ]:
raw_lazy = pl.scan_csv(
    RAW_FILE,
    schema_overrides=schema_overrides,
    null_values=["", "NA", "N/A", "NULL"],
    ignore_errors=True,
)

column_names = raw_lazy.collect_schema().names()

lowercase_mapping = {
    column: column.lower()
    for column in column_names
}

raw_lazy = raw_lazy.rename(lowercase_mapping)

print(f"Number of columns detected: {len(column_names)}")
print("CSV file was loaded as a lazy dataset.")

In [ ]:
def parse_wrds_date(column_name):
    date_text = pl.col(column_name).cast(pl.Utf8)

    return pl.coalesce([
        date_text.str.strptime(
            pl.Date,
            format="%Y/%m/%d",
            strict=False,
        ),
        date_text.str.strptime(
            pl.Date,
            format="%Y-%m-%d",
            strict=False,
        ),
        date_text.str.strptime(
            pl.Date,
            format="%m/%d/%Y",
            strict=False,
        ),
    ]).alias(column_name)


daily_clean = raw_lazy.with_columns([
    parse_wrds_date("mbrstartdt"),
    parse_wrds_date("mbrenddt"),
    parse_wrds_date("dlycaldt"),
])

daily_clean = daily_clean.filter(
    pl.col("permno").is_not_null()
    & pl.col("dlycaldt").is_not_null()
)

# Create the calendar month-end date.
daily_clean = daily_clean.with_columns(
    pl.col("dlycaldt")
    .dt.month_end()
    .alias("month")
)

# Determine whether the security was an index constituent on each day.
daily_clean = daily_clean.with_columns(
    (
        pl.col("mbrstartdt").is_not_null()
        & pl.col("mbrenddt").is_not_null()
        & (pl.col("dlycaldt") >= pl.col("mbrstartdt"))
        & (pl.col("dlycaldt") <= pl.col("mbrenddt"))
    )
    .fill_null(False)
    .alias("is_member_day")
)

print("Dates and membership indicators were created.")

In [ ]:
daily_unique = (
    daily_clean
    .group_by([
        "permno",
        "dlycaldt",
        "month",
    ])
    .agg([
        pl.col("indno").first().alias("indno"),
        pl.col("permco").first().alias("permco"),
        pl.col("hdrcusip").first().alias("hdrcusip"),
        pl.col("cusip").first().alias("cusip"),
        pl.col("ticker").first().alias("ticker"),
        pl.col("siccd").first().alias("siccd"),

        pl.col("sharetype").first().alias("sharetype"),
        pl.col("securitytype").first().alias("securitytype"),
        pl.col("securitysubtype").first().alias("securitysubtype"),
        pl.col("usincflg").first().alias("usincflg"),
        pl.col("issuertype").first().alias("issuertype"),
        pl.col("primaryexch").first().alias("primaryexch"),
        pl.col("conditionaltype").first().alias("conditionaltype"),
        pl.col("tradingstatusflg").first().alias("tradingstatusflg"),

        pl.col("dlyprc").drop_nulls().first().alias("dlyprc"),
        pl.col("dlyclose").drop_nulls().first().alias("dlyclose"),
        pl.col("dlycap").drop_nulls().first().alias("dlycap"),
        pl.col("shrout").drop_nulls().first().alias("shrout"),

        pl.col("dlyret").drop_nulls().first().alias("dlyret"),
        pl.col("dlyretx").drop_nulls().first().alias("dlyretx"),

        pl.col("dlyvol").drop_nulls().first().alias("dlyvol"),
        pl.col("dlyorddivamt").drop_nulls().first().alias("dlyorddivamt"),
        pl.col("dlynonorddivamt").drop_nulls().first().alias("dlynonorddivamt"),

        pl.col("dlyfacprc").drop_nulls().first().alias("dlyfacprc"),
        pl.col("dlycumfacpr").drop_nulls().first().alias("dlycumfacpr"),
        pl.col("dlycumfacshr").drop_nulls().first().alias("dlycumfacshr"),

        pl.col("is_member_day").max().alias("is_member_day"),

        pl.col("mbrstartdt")
        .filter(pl.col("is_member_day"))
        .min()
        .alias("active_mbrstartdt"),

        pl.col("mbrenddt")
        .filter(pl.col("is_member_day"))
        .max()
        .alias("active_mbrenddt"),
    ])
)

print("Daily security records were deduplicated.")

In [ ]:
market_month_end_lazy = (
    daily_unique
    .group_by("month")
    .agg(
        pl.col("dlycaldt")
        .max()
        .alias("market_month_end")
    )
)

daily_with_month_end = daily_unique.join(
    market_month_end_lazy,
    on="month",
    how="left",
)

daily_with_month_end = daily_with_month_end.with_columns(
    (
        pl.col("active_mbrstartdt").is_not_null()
        & pl.col("active_mbrenddt").is_not_null()
        & (
            pl.col("active_mbrstartdt")
            <= pl.col("market_month_end")
        )
        & (
            pl.col("active_mbrenddt")
            >= pl.col("market_month_end")
        )
    )
    .fill_null(False)
    .alias("is_member_month_end")
)

print("Market month-end dates were created.")

In [ ]:
monthly_lazy = (
    daily_with_month_end
    .group_by([
        "permno",
        "month",
    ])
    .agg([
        # Security identifiers
        pl.col("indno")
        .sort_by("dlycaldt")
        .drop_nulls()
        .last()
        .alias("indno"),

        pl.col("permco")
        .sort_by("dlycaldt")
        .drop_nulls()
        .last()
        .alias("permco"),

        pl.col("ticker")
        .sort_by("dlycaldt")
        .drop_nulls()
        .last()
        .alias("ticker"),

        pl.col("hdrcusip")
        .sort_by("dlycaldt")
        .drop_nulls()
        .last()
        .alias("hdrcusip"),

        pl.col("cusip")
        .sort_by("dlycaldt")
        .drop_nulls()
        .last()
        .alias("cusip"),

        pl.col("siccd")
        .sort_by("dlycaldt")
        .drop_nulls()
        .last()
        .alias("siccd"),

        pl.col("sharetype")
        .sort_by("dlycaldt")
        .drop_nulls()
        .last()
        .alias("sharetype"),

        pl.col("securitytype")
        .sort_by("dlycaldt")
        .drop_nulls()
        .last()
        .alias("securitytype"),

        pl.col("securitysubtype")
        .sort_by("dlycaldt")
        .drop_nulls()
        .last()
        .alias("securitysubtype"),

        pl.col("primaryexch")
        .sort_by("dlycaldt")
        .drop_nulls()
        .last()
        .alias("primaryexch"),

        # Market month-end date
        pl.col("market_month_end")
        .first()
        .alias("market_month_end"),

        # Monthly total return including distributions
        pl.when(pl.col("dlyret").count() > 0)
        .then(
            (
                pl.col("dlyret")
                .fill_null(0.0)
                .add(1.0)
                .product()
            ) - 1.0
        )
        .otherwise(
            pl.lit(None, dtype=pl.Float64)
        )
        .alias("monthly_return"),

        # Monthly return excluding distributions
        pl.when(pl.col("dlyretx").count() > 0)
        .then(
            (
                pl.col("dlyretx")
                .fill_null(0.0)
                .add(1.0)
                .product()
            ) - 1.0
        )
        .otherwise(
            pl.lit(None, dtype=pl.Float64)
        )
        .alias("monthly_return_ex_dividend"),

        # Month-end price
        pl.col("dlyprc")
        .sort_by("dlycaldt")
        .drop_nulls()
        .last()
        .alias("month_end_price"),

        pl.col("dlyclose")
        .sort_by("dlycaldt")
        .drop_nulls()
        .last()
        .alias("month_end_close"),

        # Month-end market capitalization
        pl.col("dlycap")
        .sort_by("dlycaldt")
        .drop_nulls()
        .last()
        .alias("month_end_market_cap"),

        # Month-end shares outstanding
        pl.col("shrout")
        .sort_by("dlycaldt")
        .drop_nulls()
        .last()
        .alias("month_end_shares_outstanding"),

        # Monthly trading volume
        pl.col("dlyvol")
        .sum()
        .alias("monthly_volume"),

        # Monthly ordinary and non-ordinary dividends
        pl.col("dlyorddivamt")
        .sum()
        .alias("monthly_ordinary_dividend"),

        pl.col("dlynonorddivamt")
        .sum()
        .alias("monthly_nonordinary_dividend"),

        # Realized volatility using daily returns
        (
            pl.col("dlyret").std(ddof=1)
            * (252.0 ** 0.5)
        )
        .alias("realized_volatility_annualized"),

        # Data availability indicators
        pl.col("dlycaldt")
        .n_unique()
        .alias("number_of_daily_records"),

        pl.col("dlyret")
        .count()
        .alias("number_of_return_observations"),

        pl.col("dlycaldt")
        .min()
        .alias("first_trading_date"),

        pl.col("dlycaldt")
        .max()
        .alias("last_trading_date"),

        # S&P 500 membership information
        pl.col("is_member_month_end")
        .max()
        .alias("is_member_month_end"),

        pl.col("active_mbrstartdt")
        .filter(pl.col("is_member_month_end"))
        .min()
        .alias("membership_start_date"),

        pl.col("active_mbrenddt")
        .filter(pl.col("is_member_month_end"))
        .max()
        .alias("membership_end_date"),

        # Adjustment factors
        pl.col("dlyfacprc")
        .sort_by("dlycaldt")
        .drop_nulls()
        .last()
        .alias("month_end_price_factor"),

        pl.col("dlycumfacpr")
        .sort_by("dlycaldt")
        .drop_nulls()
        .last()
        .alias("month_end_cumulative_price_factor"),

        pl.col("dlycumfacshr")
        .sort_by("dlycaldt")
        .drop_nulls()
        .last()
        .alias("month_end_cumulative_share_factor"),
    ])
    .sort([
        "month",
        "permno",
    ])
)

print("Monthly aggregation plan was created.")

In [ ]:
monthly_df = monthly_lazy.collect(streaming=True)

monthly_df.write_parquet(
    OUTPUT_FILE,
    compression="zstd",
)

print("Monthly dataset created successfully.")
print(f"Number of rows: {monthly_df.height:,}")
print(f"Number of columns: {monthly_df.width}")
print(f"Output file: {OUTPUT_FILE}")

In [ ]:
CSV_OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "02_crsp_sp500_monthly_2003_2025.csv"
)

monthly_df.write_csv(CSV_OUTPUT_FILE)

print(f"Full CSV file created: {CSV_OUTPUT_FILE}")

In [ ]:
duplicate_count = (
    monthly_df
    .group_by(["permno", "month"])
    .agg(pl.len().alias("row_count"))
    .filter(pl.col("row_count") > 1)
    .height
)

member_counts = (
    monthly_df
    .filter(pl.col("is_member_month_end"))
    .group_by("month")
    .agg(
        pl.col("permno")
        .n_unique()
        .alias("constituent_count")
    )
)

final_validation = monthly_df.select([
    pl.len().alias("total_rows"),
    pl.col("month").min().alias("start_month"),
    pl.col("month").max().alias("end_month"),
    pl.col("monthly_return").null_count().alias("missing_returns"),
    pl.col("month_end_price").null_count().alias("missing_prices"),
    pl.col("month_end_market_cap").null_count().alias("missing_market_caps"),
])

member_validation = member_counts.select([
    pl.col("constituent_count").min().alias("minimum_constituents"),
    pl.col("constituent_count").mean().alias("average_constituents"),
    pl.col("constituent_count").max().alias("maximum_constituents"),
])

print("Final validation summary:")
print(final_validation)

print(f"\nDuplicate PERMNO-month groups: {duplicate_count}")

print("\nMonth-end constituent count summary:")
print(member_validation)

In [ ]:
import csv
from pathlib import Path

COMPUSTAT_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "compustat"
    / "03_compustat_funda_2002_2024.csv"
)

COMPUSTAT_OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "03_compustat_funda_clean_2002_2024.parquet"
)

print(f"Compustat file exists: {COMPUSTAT_FILE.exists()}")
print(f"Compustat file: {COMPUSTAT_FILE}")
print(f"Output file: {COMPUSTAT_OUTPUT_FILE}")

if not COMPUSTAT_FILE.exists():
    raise FileNotFoundError(
        f"Compustat file not found: {COMPUSTAT_FILE}"
    )

with COMPUSTAT_FILE.open(
    mode="r",
    encoding="utf-8-sig",
    newline="",
) as file:
    reader = csv.reader(file)
    compustat_columns = next(reader)

print(f"\nNumber of columns: {len(compustat_columns)}")
print("Compustat column names:")

for number, column in enumerate(
    compustat_columns,
    start=1,
):
    print(f"{number:02d}. {column}")

In [ ]:
compustat_schema = {
    "costat": pl.Utf8,
    "curcd": pl.Utf8,
    "datafmt": pl.Utf8,
    "indfmt": pl.Utf8,
    "consol": pl.Utf8,
    "tic": pl.Utf8,
    "datadate": pl.Utf8,
    "gvkey": pl.Utf8,
    "conm": pl.Utf8,
    "cusip": pl.Utf8,
    "cik": pl.Utf8,
    "exchg": pl.Int64,
    "fyr": pl.Int64,
    "fic": pl.Utf8,
    "naics": pl.Utf8,
    "sic": pl.Utf8,
    "fyear": pl.Int64,
    "act": pl.Float64,
    "at": pl.Float64,
    "ceq": pl.Float64,
    "che": pl.Float64,
    "dlc": pl.Float64,
    "dltt": pl.Float64,
    "lct": pl.Float64,
    "lt": pl.Float64,
    "pstk": pl.Float64,
    "pstkl": pl.Float64,
    "pstkrv": pl.Float64,
    "seq": pl.Float64,
    "txditc": pl.Float64,
    "cogs": pl.Float64,
    "dp": pl.Float64,
    "ebit": pl.Float64,
    "ebitda": pl.Float64,
    "ib": pl.Float64,
    "ni": pl.Float64,
    "oiadp": pl.Float64,
    "revt": pl.Float64,
    "sale": pl.Float64,
    "xint": pl.Float64,
    "xrd": pl.Float64,
    "xsga": pl.Float64,
    "capx": pl.Float64,
    "oancf": pl.Float64,
    "csho": pl.Float64,
    "prcc_f": pl.Float64,
}

compustat_raw_df = pl.read_csv(
    COMPUSTAT_FILE,
    schema_overrides=compustat_schema,
    null_values=["", "NA", "N/A", "NULL"],
    ignore_errors=True,
)

print("Compustat data loaded successfully.")
print(f"Number of rows: {compustat_raw_df.height:,}")
print(f"Number of columns: {compustat_raw_df.width}")

In [ ]:
compustat_schema = {
    "costat": pl.Utf8,
    "curcd": pl.Utf8,
    "datafmt": pl.Utf8,
    "indfmt": pl.Utf8,
    "consol": pl.Utf8,
    "tic": pl.Utf8,
    "datadate": pl.Utf8,
    "gvkey": pl.Utf8,
    "conm": pl.Utf8,
    "cusip": pl.Utf8,
    "cik": pl.Utf8,
    "exchg": pl.Int64,
    "fyr": pl.Int64,
    "fic": pl.Utf8,
    "naics": pl.Utf8,
    "sic": pl.Utf8,
    "fyear": pl.Int64,
    "act": pl.Float64,
    "at": pl.Float64,
    "ceq": pl.Float64,
    "che": pl.Float64,
    "dlc": pl.Float64,
    "dltt": pl.Float64,
    "lct": pl.Float64,
    "lt": pl.Float64,
    "pstk": pl.Float64,
    "pstkl": pl.Float64,
    "pstkrv": pl.Float64,
    "seq": pl.Float64,
    "txditc": pl.Float64,
    "cogs": pl.Float64,
    "dp": pl.Float64,
    "ebit": pl.Float64,
    "ebitda": pl.Float64,
    "ib": pl.Float64,
    "ni": pl.Float64,
    "oiadp": pl.Float64,
    "revt": pl.Float64,
    "sale": pl.Float64,
    "xint": pl.Float64,
    "xrd": pl.Float64,
    "xsga": pl.Float64,
    "capx": pl.Float64,
    "oancf": pl.Float64,
    "csho": pl.Float64,
    "prcc_f": pl.Float64,
}

compustat_raw_df = pl.read_csv(
    COMPUSTAT_FILE,
    schema_overrides=compustat_schema,
    null_values=["", "NA", "N/A", "NULL"],
    ignore_errors=True,
)

print("Compustat data loaded successfully.")
print(f"Number of rows: {compustat_raw_df.height:,}")
print(f"Number of columns: {compustat_raw_df.width}")

In [ ]:
def parse_compustat_date(column_name):
    date_text = pl.col(column_name).cast(pl.Utf8)

    return pl.coalesce([
        date_text.str.strptime(
            pl.Date,
            format="%Y-%m-%d",
            strict=False,
        ),
        date_text.str.strptime(
            pl.Date,
            format="%Y/%m/%d",
            strict=False,
        ),
        date_text.str.strptime(
            pl.Date,
            format="%m/%d/%Y",
            strict=False,
        ),
    ]).alias(column_name)


compustat_filtered_df = (
    compustat_raw_df
    .with_columns(
        parse_compustat_date("datadate")
    )
    .filter(
        pl.col("gvkey").is_not_null()
        & pl.col("datadate").is_not_null()
        & (pl.col("datafmt") == "STD")
        & (pl.col("consol") == "C")
        & pl.col("indfmt").is_in(["INDL", "FS"])
        & (pl.col("curcd") == "USD")
        & pl.col("costat").is_in(["A", "I"])
    )
)

duplicate_groups_before = (
    compustat_filtered_df
    .group_by(["gvkey", "datadate"])
    .agg(pl.len().alias("row_count"))
    .filter(pl.col("row_count") > 1)
    .height
)

print("Compustat screening completed.")
print(f"Rows before screening: {compustat_raw_df.height:,}")
print(f"Rows after screening: {compustat_filtered_df.height:,}")
print(
    "Duplicate GVKEY-datadate groups before deduplication: "
    f"{duplicate_groups_before}"
)

In [ ]:
compustat_base_df = (
    compustat_filtered_df
    .with_columns(
        pl.when(pl.col("indfmt") == "INDL")
        .then(0)
        .otherwise(1)
        .alias("format_priority")
    )
    .sort([
        "gvkey",
        "datadate",
        "format_priority",
    ])
    .unique(
        subset=["gvkey", "datadate"],
        keep="first",
        maintain_order=True,
    )
    .drop("format_priority")
    .sort(["gvkey", "datadate"])
)

duplicate_groups_after = (
    compustat_base_df
    .group_by(["gvkey", "datadate"])
    .agg(pl.len().alias("row_count"))
    .filter(pl.col("row_count") > 1)
    .height
)

print("Duplicate records were processed.")
print(f"Rows after deduplication: {compustat_base_df.height:,}")
print(
    "Duplicate GVKEY-datadate groups after deduplication: "
    f"{duplicate_groups_after}"
)

In [ ]:
compustat_clean_df = compustat_base_df.with_columns([
    # Preferred stock: redemption value, liquidating value,
    # then carrying value.
    pl.coalesce([
        pl.col("pstkrv"),
        pl.col("pstkl"),
        pl.col("pstk"),
        pl.lit(0.0),
    ]).alias("preferred_stock"),

    # Stockholders' equity with standard fallbacks.
    pl.coalesce([
        pl.col("seq"),

        pl.when(pl.col("ceq").is_not_null())
        .then(
            pl.col("ceq")
            + pl.col("pstk").fill_null(0.0)
        ),

        pl.when(
            pl.col("at").is_not_null()
            & pl.col("lt").is_not_null()
        )
        .then(
            pl.col("at") - pl.col("lt")
        ),
    ]).alias("stockholders_equity"),

    # Revenue fallback.
    pl.coalesce([
        pl.col("revt"),
        pl.col("sale"),
    ]).alias("revenue"),

    # Earnings fallback.
    pl.coalesce([
        pl.col("ib"),
        pl.col("ni"),
    ]).alias("earnings"),

    # Total interest-bearing debt.
    (
        pl.col("dlc").fill_null(0.0)
        + pl.col("dltt").fill_null(0.0)
    ).alias("total_debt"),

    # Fiscal-year-end market capitalization in USD millions.
    (
        pl.col("prcc_f").abs()
        * pl.col("csho")
    ).alias("compustat_market_cap"),
])

In [ ]:
compustat_clean_df = compustat_clean_df.with_columns([
    (
        pl.col("stockholders_equity")
        + pl.col("txditc").fill_null(0.0)
        - pl.col("preferred_stock")
    ).alias("book_equity"),

    (
        pl.col("revenue")
        - pl.col("cogs")
        - pl.col("xsga").fill_null(0.0)
        - pl.col("xint").fill_null(0.0)
    ).alias("operating_profit"),
])

In [ ]:
compustat_clean_df = (
    compustat_clean_df
    .sort(["gvkey", "datadate"])
    .with_columns([
        pl.col("fyear")
        .shift(1)
        .over("gvkey")
        .alias("lag_fyear"),

        pl.col("at")
        .shift(1)
        .over("gvkey")
        .alias("lag_total_assets"),

        pl.col("book_equity")
        .shift(1)
        .over("gvkey")
        .alias("lag_book_equity"),
    ])
)

compustat_clean_df = compustat_clean_df.with_columns(
    (
        pl.col("fyear").is_not_null()
        & pl.col("lag_fyear").is_not_null()
        & (
            pl.col("fyear")
            - pl.col("lag_fyear")
            == 1
        )
    ).alias("is_consecutive_fiscal_year")
)

compustat_clean_df = compustat_clean_df.with_columns([
    (
        (
            pl.col("at")
            + pl.col("lag_total_assets")
        ) / 2.0
    ).alias("average_total_assets"),

    (
        (
            pl.col("book_equity")
            + pl.col("lag_book_equity")
        ) / 2.0
    ).alias("average_book_equity"),
])

In [ ]:
compustat_clean_df = compustat_clean_df.with_columns([
    # Operating profitability
    pl.when(
        pl.col("book_equity") > 0
    )
    .then(
        pl.col("operating_profit")
        / pl.col("book_equity")
    )
    .otherwise(None)
    .alias("operating_profitability"),

    # Investment factor: annual asset growth
    pl.when(
        pl.col("is_consecutive_fiscal_year")
        & (pl.col("lag_total_assets") > 0)
    )
    .then(
        pl.col("at")
        / pl.col("lag_total_assets")
        - 1.0
    )
    .otherwise(None)
    .alias("asset_growth"),

    # Accruals: earnings minus operating cash flow
    pl.when(
        pl.col("is_consecutive_fiscal_year")
        & (pl.col("average_total_assets") > 0)
    )
    .then(
        (
            pl.col("ni")
            - pl.col("oancf")
        )
        / pl.col("average_total_assets")
    )
    .otherwise(None)
    .alias("accruals"),

    # Return on equity
    pl.when(
        pl.col("is_consecutive_fiscal_year")
        & (pl.col("average_book_equity") > 0)
    )
    .then(
        pl.col("ni")
        / pl.col("average_book_equity")
    )
    .otherwise(None)
    .alias("roe"),

    # Financial leverage
    pl.when(
        pl.col("at") > 0
    )
    .then(
        pl.col("total_debt")
        / pl.col("at")
    )
    .otherwise(None)
    .alias("financial_leverage"),

    # Operating cash flow scaled by assets
    pl.when(
        pl.col("average_total_assets") > 0
    )
    .then(
        pl.col("oancf")
        / pl.col("average_total_assets")
    )
    .otherwise(None)
    .alias("operating_cash_flow_ratio"),

    # Capital expenditure scaled by assets
    pl.when(
        pl.col("at") > 0
    )
    .then(
        pl.col("capx")
        / pl.col("at")
    )
    .otherwise(None)
    .alias("capital_expenditure_ratio"),

    # Research and development scaled by assets
    pl.when(
        pl.col("at") > 0
    )
    .then(
        pl.col("xrd").fill_null(0.0)
        / pl.col("at")
    )
    .otherwise(None)
    .alias("research_and_development_ratio"),
])

In [ ]:
compustat_clean_df = compustat_clean_df.with_columns(
    pl.col("datadate")
    .dt.offset_by("6mo")
    .alias("accounting_available_date")
)

compustat_clean_df = compustat_clean_df.with_columns(
    pl.col("accounting_available_date")
    .dt.month_end()
    .alias("signal_start_month")
)

print("Accounting availability dates were created.")

In [ ]:
compustat_clean_df.write_parquet(
    COMPUSTAT_OUTPUT_FILE,
    compression="zstd",
)

print("Clean Compustat dataset created successfully.")
print(f"Number of rows: {compustat_clean_df.height:,}")
print(f"Number of columns: {compustat_clean_df.width}")
print(f"Output file: {COMPUSTAT_OUTPUT_FILE}")

In [ ]:
compustat_validation = compustat_clean_df.select([
    pl.len().alias("total_rows"),

    pl.col("gvkey")
    .n_unique()
    .alias("unique_companies"),

    pl.col("datadate")
    .min()
    .alias("start_date"),

    pl.col("datadate")
    .max()
    .alias("end_date"),

    pl.col("book_equity")
    .null_count()
    .alias("missing_book_equity"),

    (
        pl.col("book_equity") <= 0
    )
    .sum()
    .alias("nonpositive_book_equity"),

    pl.col("operating_profitability")
    .null_count()
    .alias("missing_profitability"),

    pl.col("asset_growth")
    .null_count()
    .alias("missing_asset_growth"),

    pl.col("accruals")
    .null_count()
    .alias("missing_accruals"),

    pl.col("roe")
    .null_count()
    .alias("missing_roe"),
])

remaining_duplicates = (
    compustat_clean_df
    .group_by(["gvkey", "datadate"])
    .agg(pl.len().alias("row_count"))
    .filter(pl.col("row_count") > 1)
    .height
)

print("Compustat validation summary:")
print(compustat_validation)

print(
    "\nRemaining duplicate GVKEY-datadate groups: "
    f"{remaining_duplicates}"
)

print("\nRetained reporting formats:")
print(
    compustat_clean_df
    .select([
        "indfmt",
        "datafmt",
        "consol",
        "curcd",
        "costat",
    ])
    .unique()
    .sort([
        "indfmt",
        "costat",
    ])
)

In [ ]:
COMPUSTAT_CSV_OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "03_compustat_funda_clean_2002_2024.csv"
)

compustat_clean_df.write_csv(
    COMPUSTAT_CSV_OUTPUT_FILE
)

csv_file_size_mb = (
    COMPUSTAT_CSV_OUTPUT_FILE.stat().st_size
    / (1024 ** 2)
)

print("Clean Compustat CSV file created successfully.")
print(f"Number of rows: {compustat_clean_df.height:,}")
print(f"Number of columns: {compustat_clean_df.width}")
print(f"File size: {csv_file_size_mb:.2f} MB")
print(f"Output file: {COMPUSTAT_CSV_OUTPUT_FILE}")

In [ ]:
import csv

CCM_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "crsp"
    / "04_ccm_link_table.csv"
)

CCM_PARQUET_OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "04_ccm_link_table_clean.parquet"
)

CCM_CSV_OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "04_ccm_link_table_clean.csv"
)

print(f"CCM file exists: {CCM_FILE.exists()}")
print(f"CCM file: {CCM_FILE}")

if not CCM_FILE.exists():
    raise FileNotFoundError(
        f"CCM file not found: {CCM_FILE}"
    )

with CCM_FILE.open(
    mode="r",
    encoding="utf-8-sig",
    newline="",
) as file:
    reader = csv.reader(file)
    ccm_columns = next(reader)

print(f"\nNumber of columns: {len(ccm_columns)}")
print("CCM column names:")

for number, column in enumerate(
    ccm_columns,
    start=1,
):
    print(f"{number:02d}. {column}")

In [ ]:
from datetime import date

ccm_schema = {
    "gvkey": pl.Utf8,
    "tic": pl.Utf8,
    "LINKPRIM": pl.Utf8,
    "LIID": pl.Utf8,
    "LINKTYPE": pl.Utf8,
    "LPERMNO": pl.Int64,
    "LPERMCO": pl.Int64,
    "LINKDT": pl.Utf8,
    "LINKENDDT": pl.Utf8,
}

ccm_raw_df = pl.read_csv(
    CCM_FILE,
    schema_overrides=ccm_schema,
    null_values=["", "NA", "N/A", "NULL", "E"],
    ignore_errors=True,
)

ccm_raw_df = ccm_raw_df.rename({
    column: column.lower()
    for column in ccm_raw_df.columns
})

print("CCM link table loaded successfully.")
print(f"Number of rows: {ccm_raw_df.height:,}")
print(f"Number of columns: {ccm_raw_df.width}")

In [ ]:
def parse_ccm_date(column_name):
    date_text = pl.col(column_name).cast(pl.Utf8)

    return pl.coalesce([
        date_text.str.strptime(
            pl.Date,
            format="%Y-%m-%d",
            strict=False,
        ),
        date_text.str.strptime(
            pl.Date,
            format="%Y/%m/%d",
            strict=False,
        ),
        date_text.str.strptime(
            pl.Date,
            format="%m/%d/%Y",
            strict=False,
        ),
    ]).alias(column_name)


ccm_clean_df = ccm_raw_df.with_columns([
    parse_ccm_date("linkdt"),
    parse_ccm_date("linkenddt"),

    pl.col("linktype")
    .str.strip_chars()
    .str.to_uppercase()
    .alias("linktype"),

    pl.col("linkprim")
    .str.strip_chars()
    .str.to_uppercase()
    .alias("linkprim"),
])

# Record whether the link has no stated ending date.
ccm_clean_df = ccm_clean_df.with_columns(
    pl.col("linkenddt")
    .is_null()
    .alias("is_open_ended_link")
)

# Create effective date boundaries for future matching.
ccm_clean_df = ccm_clean_df.with_columns([
    pl.col("linkdt")
    .fill_null(date(1900, 1, 1))
    .alias("link_start_date"),

    pl.col("linkenddt")
    .fill_null(date(2099, 12, 31))
    .alias("link_end_date"),
])

# Retain standard valid CRSP-Compustat links.
ccm_clean_df = ccm_clean_df.filter(
    pl.col("gvkey").is_not_null()
    & pl.col("lpermno").is_not_null()
    & pl.col("linktype").is_in(["LC", "LU"])
    & pl.col("linkprim").is_in(["P", "C"])
    & (
        pl.col("link_start_date")
        <= pl.col("link_end_date")
    )
)

print("CCM link screening completed.")
print(f"Rows before screening: {ccm_raw_df.height:,}")
print(f"Rows after screening: {ccm_clean_df.height:,}")

In [ ]:
rows_before_link_deduplication = ccm_clean_df.height

ccm_clean_df = (
    ccm_clean_df
    .unique(
        subset=[
            "gvkey",
            "lpermno",
            "linktype",
            "linkprim",
            "link_start_date",
            "link_end_date",
        ],
        keep="first",
        maintain_order=True,
    )
)

removed_duplicate_links = (
    rows_before_link_deduplication
    - ccm_clean_df.height
)

ccm_clean_df = ccm_clean_df.with_columns(
    pl.when(
        (pl.col("linktype") == "LC")
        & (pl.col("linkprim") == "P")
    )
    .then(0)
    .when(
        (pl.col("linktype") == "LC")
        & (pl.col("linkprim") == "C")
    )
    .then(1)
    .when(
        (pl.col("linktype") == "LU")
        & (pl.col("linkprim") == "P")
    )
    .then(2)
    .otherwise(3)
    .alias("link_priority")
)

ccm_clean_df = ccm_clean_df.sort([
    "lpermno",
    "link_start_date",
    "link_priority",
])

print("CCM link deduplication completed.")
print(f"Duplicate links removed: {removed_duplicate_links:,}")
print(f"Final number of links: {ccm_clean_df.height:,}")

In [ ]:
ccm_clean_df.write_parquet(
    CCM_PARQUET_OUTPUT_FILE,
    compression="zstd",
)

ccm_clean_df.write_csv(
    CCM_CSV_OUTPUT_FILE
)

print("Clean CCM link table created successfully.")
print(f"Number of rows: {ccm_clean_df.height:,}")
print(f"Number of columns: {ccm_clean_df.width}")
print(f"Parquet file: {CCM_PARQUET_OUTPUT_FILE}")
print(f"CSV file: {CCM_CSV_OUTPUT_FILE}")

In [ ]:
ccm_validation = ccm_clean_df.select([
    pl.len().alias("total_links"),

    pl.col("gvkey")
    .n_unique()
    .alias("unique_gvkeys"),

    pl.col("lpermno")
    .n_unique()
    .alias("unique_permnos"),

    pl.col("linkdt")
    .min()
    .alias("earliest_original_link_date"),

    pl.col("linkenddt")
    .max()
    .alias("latest_original_link_end_date"),

    pl.col("is_open_ended_link")
    .sum()
    .alias("open_ended_links"),

    (
        pl.col("link_start_date")
        > pl.col("link_end_date")
    )
    .sum()
    .alias("invalid_date_ranges"),
])

remaining_link_duplicates = (
    ccm_clean_df
    .group_by([
        "gvkey",
        "lpermno",
        "linktype",
        "linkprim",
        "link_start_date",
        "link_end_date",
    ])
    .agg(pl.len().alias("row_count"))
    .filter(pl.col("row_count") > 1)
    .height
)

print("CCM validation summary:")
print(ccm_validation)

print(
    "\nRemaining duplicate link groups: "
    f"{remaining_link_duplicates}"
)

print("\nLink type distribution:")
print(
    ccm_clean_df
    .group_by("linktype")
    .agg(pl.len().alias("number_of_links"))
    .sort("linktype")
)

print("\nPrimary link distribution:")
print(
    ccm_clean_df
    .group_by("linkprim")
    .agg(pl.len().alias("number_of_links"))
    .sort("linkprim")
)

In [ ]:
CRSP_MONTHLY_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "02_crsp_sp500_monthly_2003_2025.parquet"
)

CCM_CLEAN_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "04_ccm_link_table_clean.parquet"
)

CRSP_CCM_PARQUET_OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "08_crsp_sp500_monthly_ccm_linked_2003_2025.parquet"
)

CRSP_CCM_CSV_OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "08_crsp_sp500_monthly_ccm_linked_2003_2025.csv"
)

crsp_monthly_df = pl.read_parquet(
    CRSP_MONTHLY_FILE
)

ccm_link_df = pl.read_parquet(
    CCM_CLEAN_FILE
)

print("Input datasets loaded successfully.")
print(f"CRSP monthly rows: {crsp_monthly_df.height:,}")
print(f"CCM link rows: {ccm_link_df.height:,}")

In [ ]:
monthly_keys_df = (
    crsp_monthly_df
    .select([
        "permno",
        "month",
    ])
    .unique()
)

ccm_for_join_df = (
    ccm_link_df
    .select([
        "gvkey",
        "tic",
        "liid",
        "lpermno",
        "lpermco",
        "linktype",
        "linkprim",
        "link_start_date",
        "link_end_date",
        "link_priority",
    ])
    .rename({
        "tic": "ccm_ticker",
        "lpermco": "ccm_lpermco",
    })
)

valid_link_candidates_df = (
    monthly_keys_df
    .join(
        ccm_for_join_df,
        left_on="permno",
        right_on="lpermno",
        how="inner",
    )
    .filter(
        (pl.col("month") >= pl.col("link_start_date"))
        & (pl.col("month") <= pl.col("link_end_date"))
    )
)

candidate_counts_df = (
    valid_link_candidates_df
    .group_by([
        "permno",
        "month",
    ])
    .agg(
        pl.len().alias("candidate_count")
    )
)

ambiguous_groups_before_resolution = (
    candidate_counts_df
    .filter(
        pl.col("candidate_count") > 1
    )
    .height
)

print("Valid date-bounded CCM links were identified.")
print(
    "Valid link candidates: "
    f"{valid_link_candidates_df.height:,}"
)
print(
    "PERMNO-month groups with multiple candidates: "
    f"{ambiguous_groups_before_resolution:,}"
)

In [ ]:
resolved_links_df = (
    valid_link_candidates_df
    .sort(
        by=[
            "permno",
            "month",
            "link_priority",
            "link_start_date",
        ],
        descending=[
            False,
            False,
            False,
            True,
        ],
    )
    .unique(
        subset=[
            "permno",
            "month",
        ],
        keep="first",
        maintain_order=True,
    )
)

resolved_links_df = resolved_links_df.select([
    "permno",
    "month",
    "gvkey",
    "ccm_ticker",
    "ccm_lpermco",
    "liid",
    "linktype",
    "linkprim",
    "link_start_date",
    "link_end_date",
    "link_priority",
])

crsp_ccm_df = (
    crsp_monthly_df
    .join(
        resolved_links_df,
        on=[
            "permno",
            "month",
        ],
        how="left",
    )
    .sort([
        "month",
        "permno",
    ])
)

remaining_duplicates = (
    crsp_ccm_df
    .group_by([
        "permno",
        "month",
    ])
    .agg(
        pl.len().alias("row_count")
    )
    .filter(
        pl.col("row_count") > 1
    )
    .height
)

print("CRSP and CCM datasets were merged successfully.")
print(f"Final rows: {crsp_ccm_df.height:,}")
print(
    "Remaining duplicate PERMNO-month groups: "
    f"{remaining_duplicates}"
)

In [ ]:
total_monthly_rows = crsp_ccm_df.height

matched_monthly_rows = (
    crsp_ccm_df
    .filter(
        pl.col("gvkey").is_not_null()
    )
    .height
)

unmatched_monthly_rows = (
    total_monthly_rows
    - matched_monthly_rows
)

overall_match_rate = (
    matched_monthly_rows
    / total_monthly_rows
)

monthly_match_rates_df = (
    crsp_ccm_df
    .group_by("month")
    .agg(
        pl.col("gvkey")
        .is_not_null()
        .mean()
        .alias("match_rate")
    )
    .sort("month")
)

monthly_match_summary = (
    monthly_match_rates_df
    .select([
        pl.col("match_rate")
        .min()
        .alias("minimum_monthly_match_rate"),

        pl.col("match_rate")
        .mean()
        .alias("average_monthly_match_rate"),

        pl.col("match_rate")
        .max()
        .alias("maximum_monthly_match_rate"),
    ])
)

print("CRSP-CCM matching summary:")
print(f"Total monthly rows: {total_monthly_rows:,}")
print(f"Matched monthly rows: {matched_monthly_rows:,}")
print(f"Unmatched monthly rows: {unmatched_monthly_rows:,}")
print(f"Overall match rate: {overall_match_rate:.2%}")

print("\nMonthly match-rate summary:")
print(monthly_match_summary)

In [ ]:
crsp_ccm_df.write_parquet(
    CRSP_CCM_PARQUET_OUTPUT_FILE,
    compression="zstd",
)

crsp_ccm_df.write_csv(
    CRSP_CCM_CSV_OUTPUT_FILE
)

print("CRSP-CCM linked dataset created successfully.")
print(f"Number of rows: {crsp_ccm_df.height:,}")
print(f"Number of columns: {crsp_ccm_df.width}")
print(f"Parquet file: {CRSP_CCM_PARQUET_OUTPUT_FILE}")
print(f"CSV file: {CRSP_CCM_CSV_OUTPUT_FILE}")

In [ ]:
COMPUSTAT_CLEAN_PARQUET_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "03_compustat_funda_clean_2002_2024.parquet"
)

CRSP_CCM_LINKED_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "08_crsp_sp500_monthly_ccm_linked_2003_2025.parquet"
)

ACCOUNTING_PANEL_PARQUET_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "09_sp500_monthly_accounting_panel_2003_2025.parquet"
)

ACCOUNTING_PANEL_CSV_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "09_sp500_monthly_accounting_panel_2003_2025.csv"
)

crsp_ccm_df = pl.read_parquet(
    CRSP_CCM_LINKED_FILE
)

compustat_clean_df = pl.read_parquet(
    COMPUSTAT_CLEAN_PARQUET_FILE
)

print("Clean input datasets loaded successfully.")
print(f"CRSP-CCM rows: {crsp_ccm_df.height:,}")
print(f"Compustat rows: {compustat_clean_df.height:,}")

In [ ]:
compustat_for_merge_df = (
    compustat_clean_df
    .filter(
        pl.col("gvkey").is_not_null()
        & pl.col("signal_start_month").is_not_null()
    )
    .select([
        "gvkey",
        "datadate",
        "fyear",
        "fyr",
        "tic",
        "conm",
        "cusip",
        "cik",
        "fic",
        "sic",
        "naics",
        "indfmt",
        "costat",

        "accounting_available_date",
        "signal_start_month",

        "at",
        "book_equity",
        "earnings",
        "operating_profit",
        "operating_profitability",
        "asset_growth",
        "accruals",
        "roe",

        "ni",
        "ib",
        "oancf",
        "revenue",
        "total_debt",
        "financial_leverage",
        "operating_cash_flow_ratio",
        "capital_expenditure_ratio",
        "research_and_development_ratio",

        "average_total_assets",
        "average_book_equity",
        "is_consecutive_fiscal_year",

        "compustat_market_cap",
        "prcc_f",
        "csho",
    ])
    .rename({
        "tic": "compustat_ticker",
        "conm": "company_name",
        "cusip": "compustat_cusip",
        "sic": "compustat_sic",
        "naics": "compustat_naics",
        "at": "total_assets",
        "ni": "net_income",
        "ib": "income_before_extraordinary_items",
        "oancf": "operating_cash_flow",
        "prcc_f": "fiscal_year_end_price",
        "csho": "fiscal_year_end_shares_outstanding",
    })
)

duplicate_signal_groups = (
    compustat_for_merge_df
    .group_by([
        "gvkey",
        "signal_start_month",
    ])
    .agg(
        pl.len().alias("row_count")
    )
    .filter(
        pl.col("row_count") > 1
    )
    .height
)

print("Compustat merge variables were prepared.")
print(
    "Duplicate GVKEY-signal month groups before resolution: "
    f"{duplicate_signal_groups:,}"
)

In [ ]:
compustat_for_merge_df = (
    compustat_for_merge_df
    .sort([
        "gvkey",
        "signal_start_month",
        "datadate",
    ])
    .unique(
        subset=[
            "gvkey",
            "signal_start_month",
        ],
        keep="last",
        maintain_order=True,
    )
)

crsp_for_asof_df = (
    crsp_ccm_df
    .with_columns(
        pl.col("gvkey")
        .fill_null("__UNMATCHED__")
        .alias("_gvkey_join")
    )
    .sort([
        "_gvkey_join",
        "month",
    ])
)

compustat_for_asof_df = (
    compustat_for_merge_df
    .with_columns(
        pl.col("gvkey")
        .alias("_gvkey_join")
    )
    .drop("gvkey")
    .sort([
        "_gvkey_join",
        "signal_start_month",
    ])
)

print("Datasets were prepared for the point-in-time merge.")

In [ ]:
compustat_for_merge_df = (
    compustat_for_merge_df
    .sort([
        "gvkey",
        "signal_start_month",
        "datadate",
    ])
    .unique(
        subset=[
            "gvkey",
            "signal_start_month",
        ],
        keep="last",
        maintain_order=True,
    )
)

crsp_for_asof_df = (
    crsp_ccm_df
    .with_columns(
        pl.col("gvkey")
        .fill_null("__UNMATCHED__")
        .alias("_gvkey_join")
    )
    .sort([
        "_gvkey_join",
        "month",
    ])
)

compustat_for_asof_df = (
    compustat_for_merge_df
    .with_columns(
        pl.col("gvkey")
        .alias("_gvkey_join")
    )
    .drop("gvkey")
    .sort([
        "_gvkey_join",
        "signal_start_month",
    ])
)

print("Datasets were prepared for the point-in-time merge.")

In [ ]:
monthly_accounting_df = (
    crsp_for_asof_df
    .join_asof(
        compustat_for_asof_df,
        left_on="month",
        right_on="signal_start_month",
        by="_gvkey_join",
        strategy="backward",
    )
    .drop("_gvkey_join")
    .sort([
        "month",
        "permno",
    ])
)

print("Point-in-time accounting merge completed.")
print(f"Number of rows: {monthly_accounting_df.height:,}")
print(f"Number of columns: {monthly_accounting_df.width}")

In [ ]:
monthly_accounting_df = monthly_accounting_df.with_columns(
    pl.when(
        pl.col("signal_start_month").is_not_null()
    )
    .then(
        (
            pl.col("month").dt.year()
            - pl.col("signal_start_month").dt.year()
        ) * 12
        + (
            pl.col("month").dt.month()
            - pl.col("signal_start_month").dt.month()
        )
    )
    .otherwise(None)
    .alias("accounting_age_months")
)

monthly_accounting_df = monthly_accounting_df.with_columns([
    pl.col("datadate")
    .is_not_null()
    .alias("has_accounting_match"),

    (
        pl.col("accounting_age_months").is_not_null()
        & (pl.col("accounting_age_months") >= 0)
        & (pl.col("accounting_age_months") <= 12)
    )
    .fill_null(False)
    .alias("has_current_accounting_data"),
])

print("Accounting data age and validity indicators were created.")

In [ ]:
duplicate_panel_groups = (
    monthly_accounting_df
    .group_by([
        "permno",
        "month",
    ])
    .agg(
        pl.len().alias("row_count")
    )
    .filter(
        pl.col("row_count") > 1
    )
    .height
)

lookahead_violations = (
    monthly_accounting_df
    .filter(
        pl.col("signal_start_month").is_not_null()
        & (
            pl.col("signal_start_month")
            > pl.col("month")
        )
    )
    .height
)

overall_accounting_matches = (
    monthly_accounting_df
    .filter(
        pl.col("has_accounting_match")
    )
    .height
)

overall_current_accounting = (
    monthly_accounting_df
    .filter(
        pl.col("has_current_accounting_data")
    )
    .height
)

backtest_panel_df = monthly_accounting_df.filter(
    pl.col("month") >= date(2005, 1, 31)
)

backtest_accounting_matches = (
    backtest_panel_df
    .filter(
        pl.col("has_accounting_match")
    )
    .height
)

backtest_current_accounting = (
    backtest_panel_df
    .filter(
        pl.col("has_current_accounting_data")
    )
    .height
)

print("Monthly accounting panel validation:")
print(f"Total rows: {monthly_accounting_df.height:,}")
print(
    "Duplicate PERMNO-month groups: "
    f"{duplicate_panel_groups}"
)
print(
    "Look-ahead violations: "
    f"{lookahead_violations}"
)

print(
    "Overall accounting match rate: "
    f"{overall_accounting_matches / monthly_accounting_df.height:.2%}"
)

print(
    "Overall current-accounting rate: "
    f"{overall_current_accounting / monthly_accounting_df.height:.2%}"
)

print(
    "Backtest-period accounting match rate: "
    f"{backtest_accounting_matches / backtest_panel_df.height:.2%}"
)

print(
    "Backtest-period current-accounting rate: "
    f"{backtest_current_accounting / backtest_panel_df.height:.2%}"
)

In [ ]:
monthly_accounting_df.write_parquet(
    ACCOUNTING_PANEL_PARQUET_FILE,
    compression="zstd",
)

monthly_accounting_df.write_csv(
    ACCOUNTING_PANEL_CSV_FILE
)

print("Monthly accounting panel created successfully.")
print(f"Number of rows: {monthly_accounting_df.height:,}")
print(f"Number of columns: {monthly_accounting_df.width}")
print(f"Parquet file: {ACCOUNTING_PANEL_PARQUET_FILE}")
print(f"CSV file: {ACCOUNTING_PANEL_CSV_FILE}")